# Trader Performance vs. Bitcoin Market Sentiment
### Exploring how the Fear & Greed Index relates to Hyperliquid trader behavior and profitability

**Datasets**
- `historical_data.csv` — Hyperliquid historical trade fills (account, coin, price, size, side, closed PnL, fees, etc.)
- `fear_greed_index.csv` — Daily Bitcoin Fear/Greed Index (value 0–100 and classification)

**Goal:** join trades to the sentiment regime active on their trade date, then look for
differences in win rate, profitability, position sizing, and directional bias across
Fear/Greed regimes — and turn that into actionable trading-strategy insights.

> Place both CSV files in the same folder as this notebook before running.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 110

ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
COLORS = {'Extreme Fear': '#8B0000', 'Fear': '#E06666', 'Neutral': '#999999',
          'Greed': '#93C47D', 'Extreme Greed': '#1B5E20'}
PALETTE = [COLORS[c] for c in ORDER]

## 1. Load & inspect the raw data

In [3]:
trades = pd.read_csv('historical_data.csv')
fg = pd.read_csv('fear_greed_index.csv')

print("Trades shape:", trades.shape)
print("Fear/Greed shape:", fg.shape)
trades.head()

FileNotFoundError: [Errno 2] No such file or directory: 'historical_data.csv'

In [ ]:
fg.head()

In [ ]:
print(trades.dtypes)
print()
print("Unique accounts:", trades['Account'].nunique())
print("Unique coins:", trades['Coin'].nunique())
print("Side values:", trades['Side'].unique())
print("Direction values:", trades['Direction'].unique())

### Data quality note: the `Timestamp` column is unreliable

The trade file has two time fields: `Timestamp IST` (a readable string) and `Timestamp`
(supposed to be epoch milliseconds). Checking `Timestamp` shows it only takes **7 distinct
values across 211k+ rows** — it's been rounded/corrupted upstream and does not reflect
per-row trade time. `Timestamp IST` is used as the authoritative timestamp throughout this
notebook.

In [ ]:
print("Unique values in 'Timestamp' column:", trades['Timestamp'].nunique(), "(out of", len(trades), "rows)")
print(trades['Timestamp'].unique())

## 2. Clean, parse, and merge on date

In [ ]:
trades['datetime'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M')
trades['date'] = trades['datetime'].dt.date.astype(str)

fg['date'] = pd.to_datetime(fg['date']).dt.date.astype(str)
fg_map = fg.set_index('date')[['value', 'classification']]

trades = trades.merge(fg_map, on='date', how='left')

print(f"Matched sentiment for {trades['classification'].notna().sum():,} / {len(trades):,} trades")
print("Trade date range:", trades['date'].min(), "to", trades['date'].max())
trades['classification'].value_counts().reindex(ORDER)

Only rows with `Closed PnL != 0` represent a **realized** (closing) fill — rows with
`Closed PnL == 0` are position-building executions. Performance metrics below use the
realized subset; sizing/direction metrics use all trades (since risk-taking happens on
every fill, not just closes).

In [ ]:
closed = trades[trades['Closed PnL'] != 0].copy()
print(f"Realized-PnL trades: {len(closed):,} / {len(trades):,} total fills")

## 3. Sentiment regime overview

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
counts = trades['classification'].value_counts().reindex(ORDER)
ax.bar(ORDER, counts, color=PALETTE)
ax.set_title('Trade Volume (count) by Market Sentiment')
ax.set_ylabel('Number of Trades')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Trader performance by sentiment class

Win rate, profit factor (gross profit / gross loss), and average realized PnL per trade,
computed only on realized (`Closed PnL != 0`) fills.

In [ ]:
def summarize(g):
    wins = g[g['Closed PnL'] > 0]
    losses = g[g['Closed PnL'] < 0]
    gross_profit = wins['Closed PnL'].sum()
    gross_loss = -losses['Closed PnL'].sum()
    return pd.Series({
        'n_trades': len(g),
        'total_pnl': g['Closed PnL'].sum(),
        'avg_pnl': g['Closed PnL'].mean(),
        'median_pnl': g['Closed PnL'].median(),
        'win_rate_%': 100 * len(wins) / len(g) if len(g) > 0 else np.nan,
        'avg_win': wins['Closed PnL'].mean() if len(wins) > 0 else np.nan,
        'avg_loss': losses['Closed PnL'].mean() if len(losses) > 0 else np.nan,
        'profit_factor': gross_profit / gross_loss if gross_loss > 0 else np.nan,
        'avg_position_usd': g['Size USD'].mean(),
        'avg_fee': g['Fee'].mean(),
    })

perf = closed.groupby('classification').apply(summarize, include_groups=False).reindex(ORDER)
perf.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.2))
axes[0].bar(ORDER, perf['win_rate_%'], color=PALETTE)
axes[0].set_title('Win Rate by Market Sentiment'); axes[0].set_ylabel('Win Rate (%)'); axes[0].set_ylim(0,100)
for i,v in enumerate(perf['win_rate_%']): axes[0].text(i, v+1.5, f"{v:.1f}%", ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(ORDER, perf['profit_factor'], color=PALETTE)
axes[1].set_title('Profit Factor by Market Sentiment'); axes[1].set_ylabel('Gross Profit / Gross Loss')
for i,v in enumerate(perf['profit_factor']): axes[1].text(i, v+0.15, f"{v:.2f}", ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

**Observation:** performance is *not* a simple monotonic "buy fear, sell greed" story.
Win rate and profit factor peak at **Fear** (87.3% win rate, 6.66 profit factor) and
**Extreme Greed** (89.2% win rate, 11.02 profit factor) — while **Extreme Fear** (panic /
capitulation) and plain **Greed** (transitional euphoria) are comparatively the weakest
regimes for this trader cohort. Neutral sits in between on most metrics.

## 5. Position sizing (risk appetite) by sentiment

In [ ]:
size_by_sent = trades.groupby('classification')['Size USD'].agg(['mean','median','std','count']).reindex(ORDER)
size_by_sent.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.2))
ax.bar(ORDER, size_by_sent['mean'], color=PALETTE)
ax.set_title('Average Position Size (USD) by Market Sentiment'); ax.set_ylabel('Avg Position Size (USD)')
for i,v in enumerate(size_by_sent['mean']): ax.text(i, v+80, f"${v:,.0f}", ha='center', fontsize=9)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
fear_size = trades[trades['classification'].isin(['Fear','Extreme Fear'])]['Size USD']
greed_size = trades[trades['classification'].isin(['Greed','Extreme Greed'])]['Size USD']
u, p = stats.mannwhitneyu(fear_size, greed_size, alternative='two-sided')
print(f"Fear-regime avg size: ${fear_size.mean():,.2f}  |  Greed-regime avg size: ${greed_size.mean():,.2f}")
print(f"Mann-Whitney U test p-value: {p:.2e}")

**Observation:** traders in this dataset size up during **Fear** (\$7,816 avg,
combined Fear regime ≈ \$7,182) and size *down* during **Extreme Greed** (\$2,780 avg) —
the opposite of naive FOMO behavior, and highly statistically significant
(p < 0.0001). This looks like disciplined, somewhat contrarian risk-scaling rather than
crowd-following.

## 6. Long/short (directional) bias by sentiment

In [ ]:
bias = trades.groupby('classification')['Side'].apply(lambda s: (s=='BUY').mean()*100).reindex(ORDER)
fig, ax = plt.subplots(figsize=(7,4.2))
ax.bar(ORDER, bias, color=PALETTE)
ax.axhline(50, color='black', linestyle='--', linewidth=1)
ax.set_title('Buy-Side Trade Share by Market Sentiment'); ax.set_ylabel('% of Trades that are BUY')
ax.set_ylim(35,55)
for i,v in enumerate(bias): ax.text(i, v+0.3, f"{v:.1f}%", ha='center', fontsize=9)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

**Observation:** buy-side share drifts down from **51.1%** in Extreme Fear to
**44.9%** in Extreme Greed — traders lean net-buy during panic and net-sell/short during
euphoria, consistent with a contrarian tilt layered on top of the sizing behavior above.

## 7. Statistical significance tests

In [ ]:
fear_pnl = closed[closed['classification'].isin(['Fear','Extreme Fear'])]['Closed PnL']
greed_pnl = closed[closed['classification'].isin(['Greed','Extreme Greed'])]['Closed PnL']
u, p = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative='two-sided')
print(f"Fear regime: n={len(fear_pnl):,}, mean=${fear_pnl.mean():.2f}, median=${fear_pnl.median():.2f}")
print(f"Greed regime: n={len(greed_pnl):,}, mean=${greed_pnl.mean():.2f}, median=${greed_pnl.median():.2f}")
print(f"Mann-Whitney U p-value (Fear vs Greed, combined regimes): {p:.4f}  -> {'significant' if p<0.05 else 'NOT significant'}")

A binary Fear-vs-Greed split is *not* statistically significant. The real signal lives at the extremes — test that directly:

In [ ]:
ef = closed[closed['classification']=='Extreme Fear']['Closed PnL']
eg = closed[closed['classification']=='Extreme Greed']['Closed PnL']
neu = closed[closed['classification']=='Neutral']['Closed PnL']

h, p_kw = stats.kruskal(ef, eg, neu)
print(f"Kruskal-Wallis (Extreme Fear vs Neutral vs Extreme Greed): H={h:.2f}, p={p_kw:.2e}")

u2, p2 = stats.mannwhitneyu(pd.concat([ef, eg]), neu, alternative='greater')
print(f"Extremes (Ext.Fear + Ext.Greed) vs Neutral, one-sided test: p={p2:.2e}")
print(f"Extremes avg PnL: ${pd.concat([ef, eg]).mean():.2f}  vs  Neutral avg PnL: ${neu.mean():.2f}")

**Observation:** trading during sentiment *extremes* (either direction) significantly outperforms trading during Neutral sentiment (p < 0.0001).

## 8. Daily time series & correlation with the raw sentiment score

In [ ]:
daily = trades.groupby('date').agg(
    total_pnl=('Closed PnL','sum'),
    n_trades=('Closed PnL','count'),
    total_volume=('Size USD','sum'),
    sentiment_value=('value','first'),
    classification=('classification','first')
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date')
daily['cum_pnl'] = daily['total_pnl'].cumsum()

corr_same = daily[['sentiment_value','total_pnl']].corr().iloc[0,1]
daily['sentiment_value_lag1'] = daily['sentiment_value'].shift(1)
corr_lag = daily[['sentiment_value_lag1','total_pnl']].corr().iloc[0,1]
print(f"Same-day correlation (sentiment score vs daily PnL): r={corr_same:.3f}")
print(f"Lag-1 correlation (yesterday's sentiment vs today's PnL): r={corr_lag:.3f}")

**Observation:** the *linear* correlation between the raw 0–100 sentiment score and
daily PnL is weak (r ≈ -0.08 same-day, -0.11 lag-1). That's expected given the finding
above — the relationship is non-linear (best at both extremes, weaker in the middle), so
a simple linear correlation understates how useful the sentiment regime actually is.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(daily['date'], daily['cum_pnl'], color='#333333', linewidth=1.3, zorder=5)
ymin, ymax = ax.get_ylim()
for cls in ORDER:
    sub = daily[daily['classification']==cls]
    ax.scatter(sub['date'], [ymin + (ymax-ymin)*0.02]*len(sub), color=COLORS[cls], s=6, marker='s')
ax.set_title('Cumulative Realized PnL Over Time (bottom dots = daily sentiment class)')
ax.set_ylabel('Cumulative Closed PnL (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
handles = [plt.Line2D([0],[0], marker='s', color='w', markerfacecolor=COLORS[c], label=c, markersize=8) for c in ORDER]
ax.legend(handles=handles, loc='upper left', fontsize=8, ncol=3)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 9. Account-level heterogeneity: who thrives in which regime?

Aggregate stats can hide the fact that different accounts may have completely different
regime preferences. Grouping "Extreme Fear + Fear" into a **Fear regime** and
"Greed + Extreme Greed" into a **Greed regime**, restricted to accounts with 50+ realized
trades for a meaningful sample:

In [ ]:
acc_regime = closed.copy()
acc_regime['regime'] = acc_regime['classification'].map(
    {'Extreme Fear':'Fear','Fear':'Fear','Neutral':'Neutral','Greed':'Greed','Extreme Greed':'Greed'})
piv = acc_regime.groupby(['Account','regime'])['Closed PnL'].sum().unstack()
piv['n_trades'] = closed.groupby('Account').size()
piv = piv[piv['n_trades'] > 50]
piv['better_regime'] = np.where(piv['Fear'] > piv['Greed'], 'Fear', 'Greed')

print(f"{(piv['better_regime']=='Fear').sum()} / {len(piv)} accounts profit more during Fear regime")
print(f"{(piv['better_regime']=='Greed').sum()} / {len(piv)} accounts profit more during Greed regime")
piv.round(1).sort_values('n_trades', ascending=False).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(piv['Fear'], piv['Greed'], s=60, alpha=0.7, color='#4472C4', edgecolor='white')
lims = [min(piv['Fear'].min(), piv['Greed'].min())*1.1, max(piv['Fear'].max(), piv['Greed'].max())*1.1]
ax.plot(lims, lims, linestyle='--', color='gray', linewidth=1)
ax.axhline(0, color='black', linewidth=0.6); ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlabel('Total Realized PnL — Fear Regime (USD)')
ax.set_ylabel('Total Realized PnL — Greed Regime (USD)')
ax.set_title('Trader Performance: Fear Regime vs Greed Regime\n(accounts with 50+ realized trades)')
plt.tight_layout(); plt.show()

In [ ]:
overall = closed.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False)
print("Top 5 accounts by total realized PnL:")
print(overall.head().round(0))
print()
print("Bottom 5 accounts by total realized PnL:")
print(overall.tail().round(0))

**Observation:** most accounts (with a meaningful sample) are net more profitable
during **Greed** regimes than Fear — likely momentum/trend followers — while a smaller
minority are regime-specialists that consistently do better buying into **Fear**. Overall
account performance is extremely skewed: the top account cleared **~\$2.14M** in realized
PnL while the worst lost **~\$168K** — a handful of large, skilled (or well-capitalized)
accounts dominate aggregate PnL.

## 10. Symbol / coin concentration

In [ ]:
top_coins = trades.groupby('Coin')['Size USD'].sum().sort_values(ascending=False).head(10)
top_coins.round(0)

Volume is heavily concentrated in BTC, followed by HYPE, SOL, and ETH — the `@107` symbol is a Hyperliquid-specific index/perp identifier rather than a standard spot ticker.

## 11. Key Insights & Strategy Takeaways

1. **Performance is bimodal, not linear.** The cohort's win rate and profit factor peak
   during **Fear** (87.3% WR, 6.66 PF) and **Extreme Greed** (89.2% WR, 11.02 PF), and dip
   during **Extreme Fear** (76.2% WR, 2.16 PF) and plain **Greed** (76.9% WR, 3.03 PF).
   A naive "buy fear, sell greed" heuristic misses this — the transitional zones
   (moderate Greed, panic-stage Extreme Fear) are actually the weakest regimes, while
   *decisive* Fear and *runaway* Extreme Greed are where this cohort's edge shows up most.

2. **Sizing behavior is contrarian and disciplined, not FOMO-driven.** Average position
   size is largest during Fear (~\$7,800) and smallest during Extreme Greed (~\$2,780),
   a highly significant difference (p < 0.0001). Winning accounts appear to scale
   *into* fear and de-risk *into* euphoria — the opposite of retail FOMO chasing.

3. **Directional bias tilts contrarian at the extremes.** Buy-side share falls from 51.1%
   (Extreme Fear) to 44.9% (Extreme Greed), reinforcing point 2.

4. **Sentiment extremes beat the middle, statistically.** Trading during Extreme Fear or
   Extreme Greed significantly outperforms trading during Neutral sentiment
   (Kruskal-Wallis p < 0.0001) — sentiment is more useful as a *regime filter*
   (extreme vs. mid-range) than as a linear predictor (same/lag-day linear correlation
   with daily PnL is weak, r ≈ -0.08 to -0.11).

5. **Trader population splits into regime specialists.** ~31% of active accounts (50+
   trades) are net more profitable in Fear regimes; the rest do better in Greed regimes —
   suggesting two distinct strategy archetypes (contrarian/value vs. momentum/trend) are
   both represented and both viable, just under different conditions.

6. **PnL is extremely concentrated.** A small number of accounts account for the bulk of
   aggregate profit, and the worst-performing account lost ~\$168K — risk/position-sizing
   discipline (point 2) looks like a bigger differentiator between winners and losers than
   raw directional accuracy.

**Actionable strategy ideas:**
- Use the Fear/Greed Index as a **regime filter**, not a linear signal — flag Extreme Fear
  and Extreme Greed days for higher-conviction setups, and treat Neutral / transitional
  Greed as lower-edge, size-down periods.
- Scale position size *up* incrementally as fear deepens (within risk limits) and *down*
  as euphoria peaks, mirroring the behavior of this cohort's better-performing accounts.
- Segment traders/strategies into "Fear specialists" vs. "Greed specialists" for
  copy-trading or strategy-allocation products — the two groups appear genuinely distinct
  rather than one dominating universally.
